# Line Identification and Gaussian Fitting

This notebook demonstrates Milestone 3: identify a molecular transition from a small local catalog and fit a Gaussian profile to an extracted synthetic molecular-line spectrum.

## Physical meaning of fitted quantities

- **Amplitude** is the line peak above the fitted baseline, here in brightness-temperature units.
- **Centroid velocity** is the velocity of the fitted line center. It traces the bulk line-of-sight gas motion relative to the adopted rest frame.
- **Linewidth sigma** is the Gaussian velocity dispersion. It includes thermal broadening, turbulent broadening, and unresolved kinematic structure.
- **FWHM** is the full width at half maximum: `FWHM = 2 sqrt(2 ln 2) sigma`.
- **Integrated intensity** is the area under the fitted Gaussian line, `amplitude * sigma * sqrt(2 pi)`. In this notebook the fitted spectrum uses velocity in km/s and brightness temperature in K, so integrated intensity is reported in K km/s. It is the key measured quantity that later feeds LTE rotational-diagram analysis.
- **RMS noise** estimates the typical line-free fluctuation level and helps judge detection significance and fit quality.

## Synthetic demonstration data

This workflow uses a synthetic FITS cube so the repository remains reproducible without internet access or large telescope data files. In a real observation, the same steps would be applied to an externally downloaded datacube and a larger spectroscopic database.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits

from molecular_conditions.datacube import load_fits_cube
from molecular_conditions.demo_data import create_synthetic_line_cube
from molecular_conditions.fitting import fit_single_gaussian, gaussian_model
from molecular_conditions.spectra import extract_pixel_spectrum, extract_region_spectrum, estimate_rms_noise
from molecular_conditions.transitions import load_transition_table, match_transitions

DATA_DIR = PROJECT_ROOT / "data" / "demo"
FIGURE_DIR = PROJECT_ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Generate or load the demo cube

The synthetic cube header includes a CO J=1-0 rest frequency, but the spectral axis itself is radio velocity. This lets us fit the line in velocity space while identifying the molecular transition from the rest frequency.

In [ ]:
cube_path = create_synthetic_line_cube(DATA_DIR / "synthetic_line_cube.fits", overwrite=True)
cube = load_fits_cube(cube_path)
header = fits.getheader(cube_path)
rest_frequency_ghz = header["RESTFRQ"] / 1.0e9
print(f"Synthetic cube rest frequency: {rest_frequency_ghz:.7f} GHz")

## Extract peak-pixel and region-averaged spectra

The peak-pixel spectrum is useful for a high-contrast line profile. The region-averaged spectrum demonstrates the common observational practice of averaging over a core or emission region to increase signal-to-noise.

In [ ]:
spectral_axis, peak_intensity = extract_pixel_spectrum(cube, x=16, y=15)
region = {"x_min": 12, "x_max": 20, "y_min": 11, "y_max": 19}
region_axis, region_intensity = extract_region_spectrum(cube, region, statistic="mean")

velocity = spectral_axis.to(u.km / u.s)
rms = estimate_rms_noise((spectral_axis, peak_intensity), (-6300.0, -4300.0), return_quantity=True)
print(f"Peak-pixel RMS noise: {rms:.3f}")

## Fit a single Gaussian line

A single Gaussian is appropriate for this synthetic cube because the line was generated with one component. Real molecular spectra can require multiple components, baseline modeling, masking, or non-Gaussian profiles.

In [ ]:
fit = fit_single_gaussian(velocity, peak_intensity)
model = gaussian_model(velocity.value, fit.amplitude, fit.centroid, fit.sigma, fit.baseline)
residual = peak_intensity.value - model

print(f"Amplitude: {fit.amplitude:.3f} {fit.intensity_unit}")
print(f"Centroid: {fit.centroid:.3f} {fit.x_unit}")
print(f"Sigma: {fit.sigma:.3f} {fit.x_unit}")
print(f"FWHM: {fit.fwhm:.3f} {fit.x_unit}")
print(f"Baseline: {fit.baseline:.3f} {fit.intensity_unit}")
print(f"Integrated intensity: {fit.integrated_intensity:.3f} {fit.integrated_intensity_unit}")
fit.as_dict()

In [ ]:
fig, (ax, ax_resid) = plt.subplots(
    2, 1, figsize=(7.5, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)

ax.plot(velocity.value, peak_intensity.value, color="black", lw=1.4, label="Peak pixel")
ax.plot(region_axis.to_value(u.km / u.s), region_intensity.value, color="tab:blue", lw=1.1, alpha=0.65, label="Region mean")
ax.plot(velocity.value, model, color="tab:red", lw=2.0, label="Gaussian fit")
ax.axhline(fit.baseline, color="0.5", ls=":", lw=1.2, label="Fitted baseline")
ax.axvline(fit.centroid, color="tab:green", ls="--", lw=1.1)
ax.set_ylabel(f"Intensity ({peak_intensity.unit})")
ax.set_title("Synthetic CO J=1-0 Gaussian line fit")
ax.legend(loc="upper right")
ax.annotate(
    f"centroid = {fit.centroid:.2f} {fit.x_unit}\n"
    f"sigma = {fit.sigma:.2f} {fit.x_unit}\n"
    f"FWHM = {fit.fwhm:.2f} {fit.x_unit}\n"
    f"area = {fit.integrated_intensity:.2f} {fit.integrated_intensity_unit}",
    xy=(0.03, 0.95),
    xycoords="axes fraction",
    va="top",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.7"},
)

ax_resid.axhline(0.0, color="0.4", lw=1.0)
ax_resid.plot(velocity.value, residual, color="tab:purple", lw=1.2)
ax_resid.axhline(rms.value, color="tab:red", ls="--", lw=0.9)
ax_resid.axhline(-rms.value, color="tab:red", ls="--", lw=0.9)
ax_resid.set_xlabel("Radio velocity (km/s)")
ax_resid.set_ylabel("Residual")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "synthetic_gaussian_line_fit.png", dpi=150)
plt.show()

## Identify the transition

The demo catalog is intentionally tiny and local. It distinguishes the synthetic demonstration from real spectroscopic work, where one would use a more complete source such as CDMS, JPL, Splatalogue, or a project-specific curated table.

In [ ]:
catalog = load_transition_table(DATA_DIR / "transitions_demo.csv")
matches = match_transitions(rest_frequency_ghz, catalog, tolerance_mhz=5.0)
matches